# Augsburg vs Bayern (Bundesliga 2026/27, matchday 5): a pre-match forecast

Companion notebook to the [Jan's Matchday analysis](https://jhvl88.github.io/matchday/analysis/2026-27-md5-augsburg-bayern/).
It rebuilds everything in the post from public data:

1. **Dixon–Coles** — time-weighted Poisson model with the low-score correction (JAX).
2. **Bayesian shots-to-goals model** — shots on target ➜ goals, dynamic team strengths, 1. + 2. Bundesliga, sampled with NUTS (NumPyro on JAX).
3. **XGBoost** — gradient-boosted Poisson models on 30 leak-free pre-match features.
4. An equal-weight **ensemble**, a walk-forward **backtest** against bookmaker closing odds, and all charts.

**Runtime:** `Runtime ▸ Change runtime type ▸ GPU` (T4 is enough). JAX runs the NUTS chains on the GPU and XGBoost uses CUDA.
A full run takes roughly 20–40 minutes, most of it the monthly Bayesian refits in the backtest (set `RUN_BAYES_BACKTEST = False` to skip them).

**Freeze:** the forecast only uses matches played before **9 October 2026** (results through matchday 4), exactly as in the post.
MCMC is random and GPUs use different floating-point paths, so expect numbers within about a percentage point of the published ones.

In [ ]:
!nvidia-smi -L || echo "No GPU found - everything still runs on CPU, just slower"
!pip -q install numpyro adjustText
import jax; print("JAX backend:", jax.default_backend(), jax.devices())

In [ ]:
# ---------------------------------------------------------------- settings
FREEZE_DATE = "2026-10-09"        # use "2026-10-10" to also include Friday's Dortmund v Bremen
USE_EMBEDDED_2627 = True          # True = the exact matchday 1-4 rows used in the post; False = download football-data.co.uk
RUN_BAYES_BACKTEST = True
NUTS_WARMUP, NUTS_SAMPLES = 800, 800

import os, subprocess, urllib.request
for d in ["scripts", "data", "figs", "fonts", "src"]:
    os.makedirs(d, exist_ok=True)

# 25 seasons of results, shots, odds (football-data.co.uk) from a GitHub mirror, pinned to a fixed commit
url = "https://raw.githubusercontent.com/xgabora/Club-Football-Match-Data-2000-2025/25882a58a736daf7ece3781940eac17ae1117a66/data/Matches.csv"
if not os.path.exists("src/Matches.csv"):
    urllib.request.urlretrieve(url, "src/Matches.csv")

# Site fonts for the charts (optional)
for f, dst in [("ofl/dmsans/DMSans%5Bopsz,wght%5D.ttf", "fonts/DMSans.ttf")]:
    try:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/google/fonts/main/" + f, dst)
    except Exception as e:
        print("font download skipped:", e)

### 2026/27 matchdays 1–4
The mirror stops after matchday 1, so matchdays 2–4 (results, shots, shots on target, closing odds) come from
football-data.co.uk. The rows below are exactly the ones used for the post; set `USE_EMBEDDED_2627 = False` to pull the live file instead.

In [ ]:
EMBEDDED_2627 = '''
Date,HomeTeam,AwayTeam,FTHG,FTAG,HS,AS,HST,AST,AvgCH,AvgCD,AvgCA,B365H,B365D,B365A
28/08/2026,Bayern Munich,Stuttgart,5,1,21,11,9,4,1.21,7.37,9.88,1.22,7,10
29/08/2026,Elversberg,Leverkusen,3,2,11,21,6,4,5.02,4.35,1.58,5,4.5,1.57
29/08/2026,FC Koln,Hoffenheim,3,2,18,15,6,4,3.08,3.67,2.14,2.88,3.75,2.25
29/08/2026,Mainz,Paderborn,0,0,21,10,4,0,1.51,4.49,5.65,1.65,4,5
29/08/2026,RB Leipzig,M'gladbach,3,0,25,15,11,4,1.54,4.48,5.08,1.57,4.5,5.25
29/08/2026,Union Berlin,Ein Frankfurt,3,3,19,12,10,3,2.61,3.65,2.48,2.6,3.6,2.6
29/08/2026,Dortmund,Hamburg,2,0,20,8,6,3,1.32,5.49,7.99,1.3,5.75,9
30/08/2026,Freiburg,Werder Bremen,4,1,19,9,5,3,1.77,3.8,4.28,1.91,3.6,4
30/08/2026,Augsburg,Schalke 04,3,0,24,11,13,5,2.27,3.5,2.97,2.15,3.7,3.1
04/09/2026,Stuttgart,FC Koln,4,1,22,18,8,5,1.44,5.09,5.94,1.48,5,5.5
05/09/2026,Hoffenheim,Dortmund,2,3,18,11,6,4,2.43,3.81,2.59,2.88,3.9,2.2
05/09/2026,Leverkusen,Union Berlin,4,0,21,5,10,0,1.38,5.38,6.69,1.44,4.75,6.25
05/09/2026,M'gladbach,Elversberg,3,4,25,14,6,10,1.89,3.93,3.57,1.9,3.75,3.8
05/09/2026,Paderborn,Freiburg,0,1,14,16,3,5,3.58,3.8,1.92,3.6,3.6,1.95
05/09/2026,Werder Bremen,RB Leipzig,3,1,12,19,7,5,3.75,4.08,1.82,4.33,4,1.73
05/09/2026,Schalke 04,Bayern Munich,0,0,4,15,1,5,11.53,7.37,1.19,15,7.5,1.17
06/09/2026,Hamburg,Mainz,0,5,8,17,1,8,2.83,3.46,2.38,2.88,3.5,2.4
06/09/2026,Ein Frankfurt,Augsburg,1,4,22,17,8,7,1.83,4.09,3.69,1.91,3.9,3.6
11/09/2026,Union Berlin,Schalke 04,1,3,20,19,8,10,2.31,3.4,2.98,2.45,3.5,2.8
12/09/2026,Augsburg,Leverkusen,2,2,8,30,5,5,3.59,4.02,1.87,4.1,4.1,1.8
12/09/2026,Dortmund,Paderborn,3,0,14,10,8,0,1.25,6.33,9.46,1.25,6.5,10
12/09/2026,Freiburg,M'gladbach,5,0,19,5,9,0,1.68,4.18,4.4,1.67,4.33,4.5
12/09/2026,Hoffenheim,Stuttgart,2,1,20,16,8,5,2.18,3.97,2.84,2.35,4,2.63
12/09/2026,Mainz,Ein Frankfurt,1,3,18,11,5,5,1.74,4.04,4.07,1.95,4,3.4
12/09/2026,FC Koln,Werder Bremen,1,1,15,14,6,5,2.05,3.77,3.2,2.05,3.8,3.3
13/09/2026,RB Leipzig,Hamburg,5,0,19,10,6,4,1.38,5.38,6.69,1.36,5,8
13/09/2026,Elversberg,Bayern Munich,1,2,5,28,1,6,11.75,8.33,1.17,13,9,1.17
18/09/2026,Bayern Munich,Union Berlin,7,0,23,3,15,2,1.03,18,40.23,1.04,19,29
19/09/2026,Ein Frankfurt,Freiburg,2,2,10,22,3,7,2.48,3.74,2.55,2.4,3.9,2.63
19/09/2026,Hamburg,FC Koln,2,1,12,18,7,5,2.66,3.61,2.45,2.7,3.6,2.45
19/09/2026,M'gladbach,Mainz,3,4,21,17,7,8,3.17,3.69,2.11,3,3.7,2.2
19/09/2026,Werder Bremen,Augsburg,3,2,20,7,11,4,2.23,3.75,2.88,2.3,3.8,2.8
19/09/2026,Stuttgart,Dortmund,0,1,12,13,5,2,2.33,3.88,2.67,2.4,3.75,2.75
20/09/2026,Leverkusen,RB Leipzig,2,0,18,12,6,3,1.84,4.36,3.46,1.95,4.2,3.25
20/09/2026,Schalke 04,Elversberg,0,0,13,11,1,1,1.86,3.92,3.68,2,3.7,3.5
20/09/2026,Paderborn,Hoffenheim,3,1,13,23,5,2,3.84,4.25,1.77,4.33,4.5,1.67
'''

if USE_EMBEDDED_2627:
    open("data/bl_2627_md1_4.csv", "w").write(EMBEDDED_2627.strip() + "\n")
else:
    urllib.request.urlretrieve("https://www.football-data.co.uk/mmz4281/2627/D1.csv", "data/bl_2627_md1_4.csv")

## The code
Each cell below writes one module to `scripts/`. They are the same files that produced the blog post.
### Data preparation

In [ ]:
%%writefile scripts/prep.py
"""Build a clean match table: Bundesliga (D1) + 2. Bundesliga (D2), 2005/06 onwards.

Sources
- xgabora/Club-Football-Match-Data-2000-2025 (football-data.co.uk results, shots, odds + ClubElo)
- football-data.co.uk 2026/27 D1 file for matchdays 2-4 (not yet in the GitHub mirror)
"""
import sys
import numpy as np
import pandas as pd

SRC = sys.argv[1]          # path to xgabora Matches.csv
NEW = sys.argv[2]          # 2026/27 MD1-4 csv
OUT = sys.argv[3]
FREEZE = sys.argv[4] if len(sys.argv) > 4 else None   # keep only matches strictly before this date

m = pd.read_csv(SRC, low_memory=False, parse_dates=["MatchDate"])
m = m[m.Division.isin(["D1", "D2"])].copy()
m = m.rename(columns={
    "Division": "div", "MatchDate": "date", "HomeTeam": "home", "AwayTeam": "away",
    "FTHome": "hg", "FTAway": "ag", "HomeShots": "hs", "AwayShots": "as_",
    "HomeTarget": "hst", "AwayTarget": "ast", "HomeElo": "elo_h", "AwayElo": "elo_a",
    "OddHome": "odd_h", "OddDraw": "odd_d", "OddAway": "odd_a",
    "HomeRed": "hr", "AwayRed": "ar",
})
keep = ["div", "date", "home", "away", "hg", "ag", "hs", "as_", "hst", "ast",
        "hr", "ar", "elo_h", "elo_a", "odd_h", "odd_d", "odd_a"]
m = m[keep]

n = pd.read_csv(NEW)
n["date"] = pd.to_datetime(n.Date, dayfirst=True)
n = n.rename(columns={"HomeTeam": "home", "AwayTeam": "away", "FTHG": "hg", "FTAG": "ag",
                      "HS": "hs", "AS": "as_", "HST": "hst", "AST": "ast",
                      "AvgCH": "odd_h", "AvgCD": "odd_d", "AvgCA": "odd_a"})
n["div"] = "D1"
n = n[[c for c in keep if c in n.columns]]

# The mirror already holds MD1 2026/27: keep its rows (they carry Elo), append the rest.
key = lambda d: d["date"].dt.strftime("%Y-%m-%d") + "|" + d["home"] + "|" + d["away"]
n = n[~key(n).isin(set(key(m)))]
df = pd.concat([m, n], ignore_index=True)
df = df.dropna(subset=["hg", "ag"]).sort_values(["date", "div", "home"]).reset_index(drop=True)
df["season"] = np.where(df.date.dt.month >= 7, df.date.dt.year, df.date.dt.year - 1)
df = df[df.season >= 2005].reset_index(drop=True)
if FREEZE:
    df = df[df.date < FREEZE].reset_index(drop=True)
for c in ["hg", "ag"]:
    df[c] = df[c].astype(int)

df.to_csv(OUT, index=False)
d1 = df[df["div"] == "D1"]
print("rows", len(df), "| D1", len(d1), "| D2", (df["div"] == "D2").sum())
print("D1 per season:\n", d1.groupby("season").size().tail(4).to_string())
print("SOT coverage D1:", d1.hst.notna().mean().round(3), "| goals>SOT rows:", ((d1.hg > d1.hst) | (d1.ag > d1.ast)).mean().round(4))
print("2026/27 teams:", sorted(set(d1[d1.season == 2026].home)))

In [ ]:
!python scripts/prep.py src/Matches.csv data/bl_2627_md1_4.csv data/matches.csv $FREEZE_DATE

### Model 1 — Dixon–Coles (1997), written in JAX
Attack and defence ratings per club, a home edge, a 2. Bundesliga intercept and the Dixon–Coles low-score correction `rho`.
Older matches are down-weighted with `exp(-xi * days)`; `xi` is tuned on 2022/23 in the backtest (0.0019 ≈ one-year half-life).

In [ ]:
%%writefile scripts/bl_models.py
"""Model zoo for Bundesliga match prediction (JAX -> runs on GPU in Colab, CPU elsewhere)."""
import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp
from jax.scipy.special import gammaln
from scipy.optimize import minimize
from scipy.stats import poisson

jax.config.update("jax_enable_x64", True)
MAXG = 11  # score grid 0..10


# ----------------------------------------------------------------------------- utils
def team_index(df):
    teams = sorted(set(df.home) | set(df.away))
    return teams, {t: i for i, t in enumerate(teams)}


def outcome_probs(M):
    """Home / draw / away probabilities from a score matrix M[home_goals, away_goals]."""
    return np.array([np.tril(M, -1).sum(), np.trace(M), np.triu(M, 1).sum()])


def rps(p, y):
    """Ranked probability score. p: (n,3) H/D/A, y: (n,) 0=H 1=D 2=A. Lower is better."""
    p = np.atleast_2d(p)
    o = np.eye(3)[np.asarray(y)]
    cp, co = np.cumsum(p, 1)[:, :2], np.cumsum(o, 1)[:, :2]
    return ((cp - co) ** 2).sum(1) / 2


def logloss(p, y):
    p = np.clip(np.atleast_2d(p), 1e-12, 1)
    return -np.log(p[np.arange(len(p)), np.asarray(y)])


def result_code(hg, ag):
    return np.where(hg > ag, 0, np.where(hg == ag, 1, 2))


def dc_tau_matrix(lh, la, rho):
    """Dixon-Coles low-score adjustment applied to a Poisson score grid."""
    t = np.ones((MAXG, MAXG))
    t[0, 0] = 1 - lh * la * rho
    t[0, 1] = 1 + lh * rho
    t[1, 0] = 1 + la * rho
    t[1, 1] = 1 - rho
    return t


def score_matrix(lh, la, rho=0.0):
    g = np.arange(MAXG)
    M = np.outer(poisson.pmf(g, lh), poisson.pmf(g, la)) * dc_tau_matrix(lh, la, rho)
    return M / M.sum()


# ----------------------------------------------------------------------------- Dixon-Coles
class DixonColes:
    """Time-weighted Dixon-Coles (1997) with a 2. Bundesliga intercept.

    log lambda_home = mu + div + home + att[h] - def[a]
    log lambda_away = mu + div        + att[a] - def[h]
    Sum-to-zero constraints via centring; weights w = exp(-xi * days_ago).
    """

    def __init__(self, xi=0.0025, ridge=1e-3):
        self.xi, self.ridge = xi, ridge

    def fit(self, df, ref_date):
        df = df[df.date < ref_date]
        self.teams, self.idx = team_index(df)
        T = len(self.teams)
        h = jnp.array(df.home.map(self.idx).values)
        a = jnp.array(df.away.map(self.idx).values)
        x = jnp.array(df.hg.values, dtype=float)
        y = jnp.array(df.ag.values, dtype=float)
        d2 = jnp.array((df["div"] == "D2").values, dtype=float)
        days = (pd.Timestamp(ref_date) - df.date).dt.days.values
        w = jnp.array(np.exp(-self.xi * days))
        ridge = self.ridge

        def unpack(p):
            att = p[:T] - p[:T].mean()
            de = p[T:2 * T] - p[T:2 * T].mean()
            return att, de, p[2 * T], p[2 * T + 1], p[2 * T + 2], p[2 * T + 3]

        def nll(p):
            att, de, mu, home, div2, rho = unpack(p)
            lh = jnp.exp(mu + div2 * d2 + home + att[h] - de[a])
            la = jnp.exp(mu + div2 * d2 + att[a] - de[h])
            ll = x * jnp.log(lh) - lh - gammaln(x + 1) + y * jnp.log(la) - la - gammaln(y + 1)
            tau = jnp.where((x == 0) & (y == 0), 1 - lh * la * rho,
                  jnp.where((x == 0) & (y == 1), 1 + lh * rho,
                  jnp.where((x == 1) & (y == 0), 1 + la * rho,
                  jnp.where((x == 1) & (y == 1), 1 - rho, 1.0))))
            ll = ll + jnp.log(jnp.clip(tau, 1e-10, None))
            return -(w * ll).sum() / w.sum() + ridge * (att ** 2 + de ** 2).sum()

        f = jax.jit(jax.value_and_grad(nll))
        p0 = np.zeros(2 * T + 4)
        p0[2 * T] = 0.3
        p0[2 * T + 1] = 0.25
        res = minimize(lambda p: [np.asarray(v, dtype=float) for v in f(jnp.array(p))],
                       p0, jac=True, method="L-BFGS-B")
        att, de, mu, home, div2, rho = [np.asarray(v) for v in unpack(jnp.array(res.x))]
        self.att, self.de = dict(zip(self.teams, att)), dict(zip(self.teams, de))
        self.mu, self.home, self.div2, self.rho = float(mu), float(home), float(div2), float(rho)
        return self

    def rates(self, home, away, d2=False):
        base = self.mu + (self.div2 if d2 else 0.0)
        lh = np.exp(base + self.home + self.att.get(home, 0) - self.de.get(away, 0))
        la = np.exp(base + self.att.get(away, 0) - self.de.get(home, 0))
        return lh, la

    def predict_matrix(self, home, away):
        lh, la = self.rates(home, away)
        return score_matrix(lh, la, self.rho)

### Model 2 — Bayesian shots-to-goals model (NumPyro, NUTS on GPU)
Two stages per team and match: **shots on target ~ NegBin**, then **goals | shots on target ~ Binomial**.
Attack/defence (shot creation and suppression) follow random walks over half-seasons, with larger steps over the summer;
finishing and goal-prevention drift by season. Red cards and the empty-stadium season enter the fit as covariates.
Because a Binomial thinning of a NegBin is again NegBin, the predictive score grid is exact rather than simulated.

In [ ]:
%%writefile scripts/bayes_model.py
"""Bayesian dynamic 'shots-to-goals' model (NumPyro / JAX — GPU in Colab).

Two-stage data-generating story for each team in a match:
  1) chance creation:  shots on target  SOT ~ NegBin(mean = lam, conc = phi)
        log lam = alpha[div] + home*(1 - ghost) + home_ghost*ghost
                  + att[team, period] - def[opp, period] + red_eff*(red_opp - red_own)
  2) finishing:        goals | SOT ~ Binomial(SOT, p)
        logit p = beta[div] + gamma_home + fin[team, season] - keep[opp, season]
  When SOT is missing, goals ~ NegBin(mean = lam*p, conc = phi)  (thinning of a NegBin).

att/def follow Gaussian random walks over ~quarter-season periods (bigger steps over summers),
fin/keep follow slower random walks over seasons. 1. and 2. Bundesliga share one scale, so
promoted clubs carry their 2. Bundesliga form into the top flight.
"""
import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_median

PERIODS_PER_SEASON = 4


def period_of(month):
    """Within-season period index.
    4 periods: Jul-Sep=0, Oct-Dec=1, Jan-Feb=2, Mar-Jun=3.  2 periods: Jul-Dec=0, Jan-Jun=1."""
    m = np.asarray(month)
    if PERIODS_PER_SEASON == 2:
        out = np.where(m >= 7, 0, 1)
    else:
        out = np.select([(m >= 7) & (m <= 9), m >= 10, m <= 2], [0, 1, 2], 3)
    return out if out.ndim else int(out)


def build_data(df, first_season, ref_date):
    df = df[(df.season >= first_season) & (df.date < ref_date)].copy()
    teams = sorted(set(df.home) | set(df.away))
    tid = {t: i for i, t in enumerate(teams)}
    S = int(pd.Timestamp(ref_date).year - (pd.Timestamp(ref_date).month < 7) - first_season + 1)
    P = S * PERIODS_PER_SEASON
    per = (df.season.values - first_season) * PERIODS_PER_SEASON + period_of(df.date.dt.month.values)
    ghost = ((df.date >= "2020-05-16") & (df.date <= "2021-07-31")).values.astype(float)
    hst = df.hst.values.copy()
    ast = df.ast.values.copy()
    has = ~(np.isnan(hst) | np.isnan(ast))
    hst = np.where(has, np.maximum(np.nan_to_num(hst), df.hg.values), 0).astype(int)
    ast = np.where(has, np.maximum(np.nan_to_num(ast), df.ag.values), 0).astype(int)
    data = dict(
        h=df.home.map(tid).values, a=df.away.map(tid).values,
        per=per.astype(int), sea=(df.season.values - first_season).astype(int),
        d2=(df["div"] == "D2").values.astype(int), ghost=ghost,
        hst=hst, ast=ast, has=has, hg=df.hg.values.astype(int), ag=df.ag.values.astype(int),
        hr=np.nan_to_num(df.hr.values), ar=np.nan_to_num(df.ar.values),
    )
    # summer boundaries (period p where p % 4 == 0 starts a new season)
    new_season = (np.arange(P) % PERIODS_PER_SEASON == 0).astype(float)
    meta = dict(teams=teams, tid=tid, S=S, P=P, T=len(teams), new_season=new_season,
                first_season=first_season)
    return {k: jnp.asarray(v) for k, v in data.items()}, meta


def random_walk(name, T, n, sig0, sig_step):
    """Non-centred Gaussian random walk, centred across teams in every step. Returns (T, n)."""
    z = numpyro.sample(f"z_{name}", dist.Normal(0, 1).expand([T, n]).to_event(2))
    steps = z * sig_step[None, :]
    steps = steps.at[:, 0].set(z[:, 0] * sig0)
    x = jnp.cumsum(steps, axis=1)
    return x - x.mean(axis=0, keepdims=True)


def model(d, meta):
    T, P, S = meta["T"], meta["P"], meta["S"]
    new_season = jnp.asarray(meta["new_season"])

    alpha = numpyro.sample("alpha", dist.Normal(1.4, 0.5).expand([2]).to_event(1))
    beta = numpyro.sample("beta", dist.Normal(-1.0, 0.5).expand([2]).to_event(1))
    home = numpyro.sample("home", dist.Normal(0.15, 0.1))
    home_ghost = numpyro.sample("home_ghost", dist.Normal(0.0, 0.1))
    gamma_home = numpyro.sample("gamma_home", dist.Normal(0.0, 0.1))
    red_eff = numpyro.sample("red_eff", dist.Normal(0.2, 0.2))
    phi = numpyro.sample("phi", dist.LogNormal(3.0, 1.0))

    s_within = numpyro.sample("s_within", dist.HalfNormal(0.1))
    s_summer = numpyro.sample("s_summer", dist.HalfNormal(0.2))
    s_fin = numpyro.sample("s_fin", dist.HalfNormal(0.1))
    s0 = numpyro.sample("s0", dist.HalfNormal(0.4))
    s_fin0 = numpyro.sample("s_fin0", dist.HalfNormal(0.2))

    sig = s_within + (s_summer - s_within) * new_season
    att = random_walk("att", T, P, s0, sig)
    dfn = random_walk("def", T, P, s0, sig)
    sig_f = jnp.full(S, s_fin)
    fin = random_walk("fin", T, S, s_fin0, sig_f)
    keep = random_walk("keep", T, S, s_fin0, sig_f)

    h, a, per, sea, d2 = d["h"], d["a"], d["per"], d["sea"], d["d2"]
    hadv = home * (1 - d["ghost"]) + home_ghost * d["ghost"]
    eta_h = alpha[d2] + hadv + att[h, per] - dfn[a, per] + red_eff * (d["ar"] - d["hr"])
    eta_a = alpha[d2] + att[a, per] - dfn[h, per] + red_eff * (d["hr"] - d["ar"])
    lam_h = jnp.exp(jnp.clip(eta_h, -4.0, 5.0))   # numerical guard only
    lam_a = jnp.exp(jnp.clip(eta_a, -4.0, 5.0))
    p_h = jax.nn.sigmoid(beta[d2] + gamma_home + fin[h, sea] - keep[a, sea])
    p_a = jax.nn.sigmoid(beta[d2] + fin[a, sea] - keep[h, sea])

    has = d["has"]
    nb = lambda mu: dist.NegativeBinomial2(mu, phi)
    with numpyro.plate("matches", len(h)):
        with numpyro.handlers.mask(mask=has):
            numpyro.sample("hst", nb(lam_h), obs=d["hst"])
            numpyro.sample("ast", nb(lam_a), obs=d["ast"])
            numpyro.sample("hg_c", dist.Binomial(d["hst"], probs=p_h), obs=d["hg"])
            numpyro.sample("ag_c", dist.Binomial(d["ast"], probs=p_a), obs=d["ag"])
        with numpyro.handlers.mask(mask=~has):
            numpyro.sample("hg_m", nb(lam_h * p_h), obs=d["hg"])
            numpyro.sample("ag_m", nb(lam_a * p_a), obs=d["ag"])


def reconstruct(samples, meta):
    """Rebuild the latent random walks from posterior draws of the innovations."""
    T, P, S = meta["T"], meta["P"], meta["S"]
    ns = jnp.asarray(meta["new_season"])

    def rw(z, s0, sig):
        steps = z * sig[:, None, :]
        steps = steps.at[:, :, 0].set(z[:, :, 0] * s0[:, None])
        x = jnp.cumsum(steps, axis=2)
        return x - x.mean(axis=1, keepdims=True)

    sig = samples["s_within"][:, None] + (samples["s_summer"] - samples["s_within"])[:, None] * ns[None, :]
    sig_f = jnp.repeat(samples["s_fin"][:, None], S, axis=1)
    return dict(
        att=np.asarray(rw(samples["z_att"], samples["s0"], sig)),
        def_=np.asarray(rw(samples["z_def"], samples["s0"], sig)),
        fin=np.asarray(rw(samples["z_fin"], samples["s_fin0"], sig_f)),
        keep=np.asarray(rw(samples["z_keep"], samples["s_fin0"], sig_f)),
    )


def fit(df, first_season, ref_date, warmup=500, samples=500, chains=2, seed=0, chain_method=None,
        max_tree_depth=10):
    # one GPU -> run chains vectorised on it; CPU -> one chain per core
    chain_method = chain_method or ("vectorized" if jax.default_backend() == "gpu" else "parallel")
    d, meta = build_data(df, first_season, ref_date)
    kernel = NUTS(model, target_accept_prob=0.85, max_tree_depth=max_tree_depth,
                  init_strategy=init_to_median(num_samples=15))
    mcmc = MCMC(kernel, num_warmup=warmup, num_samples=samples,
                num_chains=chains, chain_method=chain_method, progress_bar=False)
    mcmc.run(jax.random.PRNGKey(seed), d, meta, extra_fields=("num_steps", "diverging"))
    return mcmc, meta


def simulate_match(samples, states, meta, home_team, away_team, ref_date, n_per_draw=20, seed=1):
    """Posterior-predictive score distribution for a future 1. Bundesliga match.

    The future period's strengths = last observed period + one random-walk innovation, so
    parameter uncertainty AND drift since the last match are both carried into the forecast.
    """
    rng = np.random.default_rng(seed)
    tid = meta["tid"]
    ref = pd.Timestamp(ref_date)
    season = ref.year - (ref.month < 7)
    p_now = (season - meta["first_season"]) * PERIODS_PER_SEASON + int(period_of(ref.month))
    p_last = min(p_now, meta["P"] - 1)
    s_idx = min(season - meta["first_season"], meta["S"] - 1)
    h, a = tid[home_team], tid[away_team]
    n = len(samples["alpha"])
    # The forecast period (Oct-Dec 2026) has no matches yet, so its latent state is the
    # Jul-Sep state plus one posterior random-walk innovation: drift uncertainty is included.
    att, de, fin, keep = states["att"], states["def_"], states["fin"], states["keep"]
    lam_h = np.exp(samples["alpha"][:, 0] + samples["home"] + att[:, h, p_last] - de[:, a, p_last])
    lam_a = np.exp(samples["alpha"][:, 0] + att[:, a, p_last] - de[:, h, p_last])
    ph = 1 / (1 + np.exp(-(samples["beta"][:, 0] + samples["gamma_home"] + fin[:, h, s_idx] - keep[:, a, s_idx])))
    pa = 1 / (1 + np.exp(-(samples["beta"][:, 0] + fin[:, a, s_idx] - keep[:, h, s_idx])))
    phi = samples["phi"]

    def nb_draw(mu):
        g = rng.gamma(np.repeat(phi, n_per_draw), np.repeat(mu / phi, n_per_draw))
        return rng.poisson(g)

    sot_h, sot_a = nb_draw(lam_h), nb_draw(lam_a)
    g_h = rng.binomial(sot_h, np.repeat(ph, n_per_draw))
    g_a = rng.binomial(sot_a, np.repeat(pa, n_per_draw))
    return dict(lam_sot_h=lam_h, lam_sot_a=lam_a, p_h=ph, p_a=pa,
                xg_h=lam_h * ph, xg_a=lam_a * pa, sot_h=sot_h, sot_a=sot_a, g_h=g_h, g_a=g_a)


def predict_matrix(samples, states, meta, home_team, away_team, date, maxg=11, thin=4):
    """Exact posterior-predictive score matrix: goals = Binomial thinning of NegBin shots on target
    => goals ~ NegBin(mean = lam * p, conc = phi); averaged over posterior draws."""
    from scipy.stats import nbinom
    tid = meta["tid"]
    ref = pd.Timestamp(date)
    season = ref.year - (ref.month < 7)
    p_now = min((season - meta["first_season"]) * PERIODS_PER_SEASON + int(period_of(ref.month)), meta["P"] - 1)
    s_idx = min(season - meta["first_season"], meta["S"] - 1)
    sl = slice(None, None, thin)
    al, be = samples["alpha"][sl, 0], samples["beta"][sl, 0]
    att, de = states["att"][sl], states["def_"][sl]
    fin, keep = states["fin"][sl], states["keep"][sl]
    phi = samples["phi"][sl]
    out = np.zeros((maxg, maxg))
    if home_team not in tid or away_team not in tid:
        return None
    h, a = tid[home_team], tid[away_team]
    lam_h = np.exp(al + samples["home"][sl] + att[:, h, p_now] - de[:, a, p_now])
    lam_a = np.exp(al + att[:, a, p_now] - de[:, h, p_now])
    ph = 1 / (1 + np.exp(-(be + samples["gamma_home"][sl] + fin[:, h, s_idx] - keep[:, a, s_idx])))
    pa = 1 / (1 + np.exp(-(be + fin[:, a, s_idx] - keep[:, h, s_idx])))
    g = np.arange(maxg)
    mh, ma = lam_h * ph, lam_a * pa
    fh = nbinom.pmf(g[None, :], phi[:, None], (phi / (phi + mh))[:, None])
    fa = nbinom.pmf(g[None, :], phi[:, None], (phi / (phi + ma))[:, None])
    out = np.einsum("ni,nj->ij", fh, fa) / len(phi)
    return out / out.sum()

### Model 3 — leak-free features + XGBoost (CUDA)

In [ ]:
%%writefile scripts/features.py
"""Leak-free pre-match features: own goal-based Elo + exponentially weighted team form.

Every feature for a match only uses matches strictly before its kick-off date.
1. and 2. Bundesliga share one timeline, so promoted clubs arrive with real form.
"""
import numpy as np
import pandas as pd


def elo_ratings(df, k=20.0, home_adv=60.0, carry=0.8, d2_offset=-150.0):
    """World-Football-Elo style ratings with a goal-difference multiplier.

    carry: share of the deviation from the league mean kept over the summer.
    d2_offset: starting rating gap for clubs first seen in the 2. Bundesliga.
    Returns pre-match ratings (elo_h, elo_a) aligned with df's rows.
    """
    r = {}
    out_h, out_a = np.empty(len(df)), np.empty(len(df))
    last_season = None
    for i, row in enumerate(df.itertuples(index=False)):
        if row.season != last_season and last_season is not None:
            mean = np.mean(list(r.values()))
            r = {t: mean + carry * (v - mean) for t, v in r.items()}
        last_season = row.season
        base = 1500.0 + (d2_offset if row.div == "D2" else 0.0)
        rh, ra = r.setdefault(row.home, base), r.setdefault(row.away, base)
        out_h[i], out_a[i] = rh, ra
        if np.isnan(row.hg):          # future fixture: rating only, no update
            continue
        exp_h = 1 / (1 + 10 ** (-(rh + home_adv - ra) / 400))
        gd = row.hg - row.ag
        res = 1.0 if gd > 0 else 0.5 if gd == 0 else 0.0
        mult = 1.0 if abs(gd) <= 1 else (1.5 if abs(gd) == 2 else (11 + abs(gd)) / 8)
        delta = k * mult * (res - exp_h)
        r[row.home], r[row.away] = rh + delta, ra - delta
    return out_h, out_a


def team_long(df):
    """One row per team per match (for rolling form)."""
    base = dict(date=df.date.values, mid=df.index.values, season=df.season.values)
    home = pd.DataFrame({**base, "team": df.home.values, "opp": df.away.values, "is_home": 1,
                         "gf": df.hg.values, "ga": df.ag.values, "sf": df.hs.values, "sa": df.as_.values,
                         "tf": df.hst.values, "ta": df.ast.values})
    away = pd.DataFrame({**base, "team": df.away.values, "opp": df.home.values, "is_home": 0,
                         "gf": df.ag.values, "ga": df.hg.values, "sf": df.as_.values, "sa": df.hs.values,
                         "tf": df.ast.values, "ta": df.hst.values})
    return pd.concat([home, away]).sort_values(["team", "date", "mid"]).reset_index(drop=True)


def ewm_form(long, halflife=8, stats=("gf", "ga", "sf", "sa", "tf", "ta")):
    """EWMA of each stat over the team's previous matches (shifted => strictly pre-match)."""
    out = long[["mid", "team", "date"]].copy()
    g = long.groupby("team", sort=False)
    for s in stats:
        out[f"{s}_ewm"] = g[s].transform(lambda x: x.shift(1).ewm(halflife=halflife, ignore_na=True).mean())
    out["n_prev"] = g.cumcount()
    out["rest_days"] = g["date"].transform(lambda x: x.diff().dt.days).clip(upper=30)
    out["pts_last5"] = g.apply(lambda x: pd.Series(
        np.where(x.gf > x.ga, 3, np.where(x.gf == x.ga, 1, 0)), index=x.index
    ).shift(1).rolling(5, min_periods=1).sum()).reset_index(level=0, drop=True)
    return out


def build_features(df, halflife=8):
    df = df.sort_values(["date", "div", "home"]).reset_index(drop=True)
    df["elo_h_own"], df["elo_a_own"] = elo_ratings(df)
    long = team_long(df)
    form = ewm_form(long, halflife)
    is_home = long.is_home.values == 1
    fh = form[is_home].set_index("mid").add_prefix("h_")
    fa = form[~is_home].set_index("mid").add_prefix("a_")
    X = df.join(fh, how="left").join(fa, how="left")
    X["elo_diff"] = X.elo_h_own - X.elo_a_own
    for s in ["gf", "ga", "sf", "sa", "tf", "ta"]:
        X[f"d_{s}"] = X[f"h_{s}_ewm"] - X[f"a_{s}_ewm"]
    X["h_conv"] = X.h_gf_ewm / X.h_tf_ewm.clip(lower=0.5)
    X["a_conv"] = X.a_gf_ewm / X.a_tf_ewm.clip(lower=0.5)
    X["h_keep"] = X.h_ga_ewm / X.h_ta_ewm.clip(lower=0.5)
    X["a_keep"] = X.a_ga_ewm / X.a_ta_ewm.clip(lower=0.5)
    # promoted this season = first top-flight season after a 2. Bundesliga campaign
    d1_seasons = {}
    for s, g in df[df["div"] == "D1"].groupby("season"):
        d1_seasons[s] = set(g.home)
    X["h_promoted"] = [int(h not in d1_seasons.get(s - 1, set())) for h, s in zip(X.home, X.season)]
    X["a_promoted"] = [int(a not in d1_seasons.get(s - 1, set())) for a, s in zip(X.away, X.season)]
    X["ghost"] = ((X.date >= "2020-05-16") & (X.date <= "2021-07-31")).astype(int)
    X["y"] = np.where(X.hg > X.ag, 0, np.where(X.hg == X.ag, 1, 2))
    return X


FEATURES = [
    "elo_h_own", "elo_a_own", "elo_diff",
    "h_gf_ewm", "h_ga_ewm", "h_sf_ewm", "h_sa_ewm", "h_tf_ewm", "h_ta_ewm",
    "a_gf_ewm", "a_ga_ewm", "a_sf_ewm", "a_sa_ewm", "a_tf_ewm", "a_ta_ewm",
    "d_gf", "d_ga", "d_tf", "d_ta", "h_conv", "a_conv", "h_keep", "a_keep",
    "h_pts_last5", "a_pts_last5", "h_rest_days", "a_rest_days",
    "h_promoted", "a_promoted", "ghost",
]

In [ ]:
%%writefile scripts/backtest_utils.py
import xgboost as xgb
from features import FEATURES


def xgb_params(device):
    return dict(objective="count:poisson", max_depth=3, learning_rate=0.03, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=20, reg_lambda=5.0, tree_method="hist",
                device=device, max_delta_step=0.7)


def train_xgb_pair(train, device=None, rounds=None):
    """Two Poisson boosters (home goals, away goals). Early-stops on the last training season."""
    import shutil
    device = device or ("cuda" if shutil.which("nvidia-smi") else "cpu")   # GPU boosting in Colab
    last = train.season.max()
    tr, va = train[train.season < last], train[train.season == last]
    models, best = {}, {}
    for tgt in ["hg", "ag"]:
        dtr = xgb.DMatrix(tr[FEATURES], label=tr[tgt])
        dva = xgb.DMatrix(va[FEATURES], label=va[tgt])
        if rounds is None:
            b = xgb.train(xgb_params(device), dtr, 3000, evals=[(dva, "va")],
                          early_stopping_rounds=150, verbose_eval=False)
            best[tgt] = b.best_iteration + 1
        else:
            best[tgt] = rounds[tgt]
        dall = xgb.DMatrix(train[FEATURES], label=train[tgt])
        models[tgt] = xgb.train(xgb_params(device), dall, best[tgt])
    return models, best

## Walk-forward backtest
Dixon–Coles is refitted every week, XGBoost once per season (trained only on earlier seasons), the Bayesian model every month.
Every forecast uses only matches played before it. Bookmaker closing odds (margin removed proportionally) are the benchmark.

In [ ]:
%%writefile scripts/backtest.py
"""Walk-forward backtest: Dixon-Coles (refit weekly), XGBoost-Poisson (refit per season), market odds.

Writes data/oos_preds.parquet with out-of-sample score-matrix-based H/D/A probabilities.
"""
import sys, time
sys.path.insert(0, "scripts")
import numpy as np
import pandas as pd
import xgboost as xgb
from bl_models import DixonColes, score_matrix, outcome_probs, rps, logloss, MAXG
from features import build_features, FEATURES
from backtest_utils import train_xgb_pair

TEST_SEASONS = [2022, 2023, 2024, 2025, 2026]   # 2022 = validation season for tuning
df = pd.read_csv("data/matches.csv", parse_dates=["date"])

# --------------------------------------------------------------------------- features
X = build_features(df)
X.to_pickle("data/features.pkl")
d1 = X[(X["div"] == "D1")].copy()
d1["week"] = d1.date.dt.to_period("W-MON").dt.start_time   # Tue..Mon blocks


# --------------------------------------------------------------------------- Dixon-Coles
def dc_walk_forward(xi, seasons):
    rows = []
    hist = df[df.season >= 2014]
    for s in seasons:
        test = d1[d1.season == s]
        for wk, g in test.groupby("week"):
            ref = g.date.min()
            m = DixonColes(xi=xi).fit(hist, ref)
            for r in g.itertuples():
                lh, la = m.rates(r.home, r.away)
                M = score_matrix(lh, la, m.rho)
                rows.append(dict(idx=r.Index, dc_lh=lh, dc_la=la, dc_rho=m.rho,
                                 **dict(zip(["dc_H", "dc_D", "dc_A"], outcome_probs(M)))))
    return pd.DataFrame(rows).set_index("idx")


t = time.time()
tune = {}
for xi in [0.0012, 0.0019, 0.0028, 0.004]:
    p = dc_walk_forward(xi, [2022])
    y = d1.loc[p.index, "y"].values
    tune[xi] = rps(p[["dc_H", "dc_D", "dc_A"]].values, y).mean()
    print(f"DC xi={xi}: RPS 2022/23 = {tune[xi]:.4f}", flush=True)
best_xi = min(tune, key=tune.get)
print("best xi", best_xi, f"({time.time()-t:.0f}s)", flush=True)
dc = dc_walk_forward(best_xi, TEST_SEASONS)


# --------------------------------------------------------------------------- XGBoost-Poisson (refit per season)
rows = []
train_pool = d1[(d1.season >= 2008) & (d1.h_n_prev >= 5) & (d1.a_n_prev >= 5)]
for s in TEST_SEASONS:
    train = train_pool[train_pool.season < s]
    models, best = train_xgb_pair(train)
    test = d1[d1.season == s]
    lh = models["hg"].predict(xgb.DMatrix(test[FEATURES]))
    la = models["ag"].predict(xgb.DMatrix(test[FEATURES]))
    rho = dc.loc[test.index, "dc_rho"].values
    for i, idx in enumerate(test.index):
        M = score_matrix(lh[i], la[i], rho[i])
        rows.append(dict(idx=idx, xgb_lh=lh[i], xgb_la=la[i],
                         **dict(zip(["xgb_H", "xgb_D", "xgb_A"], outcome_probs(M)))))
    print(f"XGB season {s}: rounds {best}", flush=True)
xg = pd.DataFrame(rows).set_index("idx")

# --------------------------------------------------------------------------- market
mk = d1[["odd_h", "odd_d", "odd_a"]].copy()
inv = 1 / mk
mk[["mkt_H", "mkt_D", "mkt_A"]] = inv.div(inv.sum(1), axis=0).values  # proportional de-vig

out = d1.loc[dc.index, ["date", "season", "home", "away", "hg", "ag", "y", "elo_diff"]] \
    .join(dc).join(xg).join(mk[["mkt_H", "mkt_D", "mkt_A"]])
out.to_pickle("data/oos_preds.pkl")
for s in TEST_SEASONS:
    o = out[out.season == s]
    msg = " | ".join(f"{m}: RPS {rps(o[[f'{m}_H', f'{m}_D', f'{m}_A']].values, o.y).mean():.4f}"
                     for m in ["dc", "xgb", "mkt"])
    print(s, len(o), msg)
print("saved", len(out), "best_xi", best_xi)

In [ ]:
!python scripts/backtest.py

In [ ]:
%%writefile scripts/bayes_backtest.py
"""Walk-forward backtest of the Bayesian model: monthly NUTS refits through 2025/26 (+2026/27 MD1-4)."""
import sys, time, pickle
import numpyro
numpyro.set_host_device_count(2)
sys.path.insert(0, "scripts")
import numpy as np, pandas as pd
import bayes_model as bm
from bl_models import outcome_probs
bm.PERIODS_PER_SEASON = 2
warm, ns = int(sys.argv[1]), int(sys.argv[2])
df = pd.read_csv("data/matches.csv", parse_dates=["date"])
oos = pd.read_pickle("data/oos_preds.pkl")
test = oos[oos.season.isin([2025, 2026])].sort_values("date")
months = test.date.dt.to_period("M")
rows = []
for mth, g in test.groupby(months):
    ref = g.date.min().normalize()
    t = time.time()
    mcmc, meta = bm.fit(df, ref.year - (ref.month < 7) - 6, ref, warmup=warm, samples=ns, chains=2, max_tree_depth=8)
    s = {k: np.asarray(v) for k, v in mcmc.get_samples().items()}
    st = bm.reconstruct(s, meta)
    for r in g.itertuples():
        M = bm.predict_matrix(s, st, meta, r.home, r.away, r.date)
        lh = (M.sum(1) * np.arange(M.shape[0])).sum(); la = (M.sum(0) * np.arange(M.shape[1])).sum()
        rows.append(dict(idx=r.Index, bay_lh=lh, bay_la=la, **dict(zip(["bay_H", "bay_D", "bay_A"], outcome_probs(M)))))
    print(mth, len(g), f"{time.time()-t:.0f}s", flush=True)
    pd.DataFrame(rows).set_index("idx").to_pickle("data/oos_bayes.pkl")
print("done")

In [ ]:
if RUN_BAYES_BACKTEST:
    !python scripts/bayes_backtest.py 250 250
else:
    print("Skipped - analysis.py needs data/oos_bayes.pkl for the backtest charts.")

## Fit the final models and forecast Augsburg v Bayern

In [ ]:
%%writefile scripts/run_bayes.py
import sys, time, pickle
import numpyro
numpyro.set_host_device_count(2)
sys.path.insert(0, "scripts")
import numpy as np, pandas as pd, jax
import bayes_model
from bayes_model import fit, reconstruct, simulate_match
bayes_model.PERIODS_PER_SEASON = int(sys.argv[5]) if len(sys.argv) > 5 else 4
first, warm, ns, out = int(sys.argv[1]), int(sys.argv[2]), int(sys.argv[3]), sys.argv[4]
df = pd.read_csv("data/matches.csv", parse_dates=["date"])
ref = "2026-10-10"
t = time.time()
mcmc, meta = fit(df, first, ref, warmup=warm, samples=ns, chains=2, max_tree_depth=int(sys.argv[6]) if len(sys.argv) > 6 else 10)
print("tree depth (mean num_steps):", float(np.asarray(mcmc.get_extra_fields()["num_steps"]).mean()), flush=True)
print(f"fit {time.time()-t:.0f}s", flush=True)
s = {k: np.asarray(v) for k, v in mcmc.get_samples().items()}
extra = mcmc.get_extra_fields()
print("divergences:", int(np.asarray(extra["diverging"]).sum()))
summ = numpyro.diagnostics.summary(mcmc.get_samples(group_by_chain=True))
for k in ["alpha","beta","home","home_ghost","gamma_home","red_eff","phi","s_within","s_summer","s_fin","s0","s_fin0"]:
    v = summ[k]; print(k, np.round(v["mean"],3), "rhat", np.round(v["r_hat"],3), "neff", np.round(v["n_eff"],0))
rh = np.concatenate([np.ravel(summ[k]["r_hat"]) for k in summ]); print("max rhat all:", np.nanmax(rh).round(3), "frac>1.05:", (rh>1.05).mean().round(4))
st = reconstruct({k: np.asarray(v) for k, v in mcmc.get_samples().items()}, meta)
sim = simulate_match(s, st, meta, "Augsburg", "Bayern Munich", ref, n_per_draw=40)
gh, ga = sim["g_h"], sim["g_a"]
print("xG FCA %.2f FCB %.2f | SOT FCA %.1f FCB %.1f | conv FCA %.3f FCB %.3f" % (sim["xg_h"].mean(), sim["xg_a"].mean(), sim["lam_sot_h"].mean(), sim["lam_sot_a"].mean(), sim["p_h"].mean(), sim["p_a"].mean()))
print("H/D/A", np.round([(gh>ga).mean(), (gh==ga).mean(), (gh<ga).mean()],3))
pickle.dump(dict(samples=s, states=st, meta=meta, sim=sim), open(out, "wb"))

In [ ]:
!python scripts/run_bayes.py 2020 $NUTS_WARMUP $NUTS_SAMPLES data/nuts_final.pkl 2 8

In [ ]:
%%writefile scripts/predict_final.py
"""Pre-match forecasts for FC Augsburg v FC Bayern (10 Oct 2026) from Dixon-Coles and XGBoost."""
import sys, pickle
sys.path.insert(0, "scripts")
import numpy as np, pandas as pd, xgboost as xgb
from bl_models import DixonColes, score_matrix, outcome_probs
from features import build_features, FEATURES
from backtest_utils import train_xgb_pair

HOME, AWAY, DATE = "Augsburg", "Bayern Munich", pd.Timestamp("2026-10-10")
df = pd.read_csv("data/matches.csv", parse_dates=["date"])
fx = pd.DataFrame([dict(div="D1", date=DATE, home=HOME, away=AWAY, hg=np.nan, ag=np.nan, season=2026)])
X = build_features(pd.concat([df, fx], ignore_index=True))
row = X[(X.home == HOME) & (X.away == AWAY) & (X.date == DATE)]

dc = DixonColes(xi=0.0019).fit(df[df.season >= 2014], DATE)
lh_dc, la_dc = dc.rates(HOME, AWAY)
M_dc = score_matrix(lh_dc, la_dc, dc.rho)

d1 = X[(X["div"] == "D1") & X.hg.notna()]
pool = d1[(d1.season >= 2008) & (d1.h_n_prev >= 5) & (d1.a_n_prev >= 5)]
models, best = train_xgb_pair(pool)
dm = xgb.DMatrix(row[FEATURES])
lh_x, la_x = float(models["hg"].predict(dm)[0]), float(models["ag"].predict(dm)[0])
M_x = score_matrix(lh_x, la_x, dc.rho)
shap_h = models["hg"].predict(dm, pred_contribs=True)[0]
shap_a = models["ag"].predict(dm, pred_contribs=True)[0]
gain_h = models["hg"].get_score(importance_type="gain")
gain_a = models["ag"].get_score(importance_type="gain")

print("DC  lam %.2f-%.2f  H/D/A %s rho %.3f" % (lh_dc, la_dc, np.round(outcome_probs(M_dc), 3), dc.rho))
print("XGB lam %.2f-%.2f  H/D/A %s  rounds %s" % (lh_x, la_x, np.round(outcome_probs(M_x), 3), best))
print(row[["elo_h_own", "elo_a_own", "h_tf_ewm", "h_ta_ewm", "a_tf_ewm", "a_ta_ewm", "h_gf_ewm", "a_gf_ewm", "h_rest_days", "a_rest_days"]].T)
# Dixon-Coles ratings table for the current 18 clubs
cur = sorted(set(df[(df.season == 2026) & (df["div"] == "D1")].home))
ratings = pd.DataFrame({"team": cur, "att": [dc.att[t] for t in cur], "def": [dc.de[t] for t in cur]})
pickle.dump(dict(M_dc=M_dc, M_x=M_x, lam_dc=(lh_dc, la_dc), lam_x=(lh_x, la_x), rho=dc.rho,
                 shap_h=shap_h, shap_a=shap_a, features=FEATURES, row=row, gain_h=gain_h, gain_a=gain_a,
                 dc_ratings=ratings, dc_home=dc.home, dc_mu=dc.mu), open("data/final_dc_xgb.pkl", "wb"))

In [ ]:
!python scripts/predict_final.py

## Ensemble, evaluation and charts

In [ ]:
%%writefile scripts/analysis.py
"""Combine models, evaluate, and draw every chart for the blog post."""
import sys, json, pickle
sys.path.insert(0, "scripts")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
import bayes_model as bm
from bl_models import outcome_probs, rps, logloss

bm.PERIODS_PER_SEASON = 2
OUT = sys.argv[1] if len(sys.argv) > 1 else "figs"
HOME, AWAY, DATE = "Augsburg", "Bayern Munich", "2026-10-10"

# ----------------------------------------------------------------------------- style
for f in ["fonts/DMSans.ttf"]:
    try:
        font_manager.fontManager.addfont(f)
        plt.rcParams["font.family"] = font_manager.FontProperties(fname=f).get_name()
    except Exception:
        pass
INK, MUTED, GRID, SURF = "#16161c", "#5f5f69", "#e6e6ea", "#ffffff"
FCA, FCB, DRAW, NEUTRAL = "#008300", "#e34948", "#9a9aa4", "#bdbdc5"
BLUE_RAMP = ["#f6f9fe", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "font.size": 10.5, "axes.titlesize": 11, "svg.fonttype": "path",
    "axes.titlelocation": "left", "axes.titleweight": "normal", "axes.titlecolor": INK,
})


def save(fig, name):
    fig.savefig(f"{OUT}/{name}.svg", bbox_inches="tight", pad_inches=0.15)
    fig.savefig(f"{OUT}/{name}.png", dpi=200, bbox_inches="tight", pad_inches=0.15)
    plt.close(fig)


# ----------------------------------------------------------------------------- load models
fd = pickle.load(open("data/final_dc_xgb.pkl", "rb"))
bay = pickle.load(open("data/nuts_final.pkl", "rb"))
S, ST, META = bay["samples"], bay["states"], bay["meta"]
M_dc, M_x = fd["M_dc"], fd["M_x"]
M_bay = bm.predict_matrix(S, ST, META, HOME, AWAY, DATE, thin=1)
M_ens = (M_dc + M_bay + M_x) / 3                    # pre-registered equal weights
odds = np.array([12.0, 8.5, 1.17])                  # bookmaker 1X2 on the morning of the match
mkt = (1 / odds) / (1 / odds).sum()

g = np.arange(M_ens.shape[0])
summ = {}
for name, M in [("Dixon-Coles", M_dc), ("Bayesian", M_bay), ("XGBoost", M_x), ("Ensemble", M_ens)]:
    p = outcome_probs(M)
    i, j = np.unravel_index(M.argmax(), M.shape)
    summ[name] = dict(H=p[0], D=p[1], A=p[2], xg_h=(M.sum(1) * g).sum(), xg_a=(M.sum(0) * g).sum(),
                      mode=f"{i}-{j}", p_mode=M[i, j])
summ["Bookmakers"] = dict(H=mkt[0], D=mkt[1], A=mkt[2])
tot = np.add.outer(g, g)
extra = dict(
    over25=M_ens[tot > 2.5].sum(), over35=M_ens[tot > 3.5].sum(), btts=M_ens[1:, 1:].sum(),
    fca_clean=M_ens[:, 0].sum(), fcb_clean=M_ens[0, :].sum(), fcb_by2=sum(M_ens[i, j] for i in g for j in g if j - i >= 2),
    top_scores=[(f"{i}-{j}", float(M_ens[i, j])) for i, j in zip(*np.unravel_index(np.argsort(M_ens, axis=None)[::-1][:6], M_ens.shape))],
    mkt_overround=float((1 / odds).sum() - 1),
)
sim = bm.simulate_match(S, ST, META, HOME, AWAY, DATE, n_per_draw=40)
extra.update(bay_sot_h=float(sim["lam_sot_h"].mean()), bay_sot_a=float(sim["lam_sot_a"].mean()),
             bay_conv_h=float(sim["p_h"].mean()), bay_conv_a=float(sim["p_a"].mean()),
             bay_xg_h_ci=np.percentile(sim["xg_h"], [10, 90]).tolist(),
             bay_xg_a_ci=np.percentile(sim["xg_a"], [10, 90]).tolist())

# ----------------------------------------------------------------------------- backtest
oos = pd.read_pickle("data/oos_preds.pkl").join(pd.read_pickle("data/oos_bayes.pkl"), how="left")
cols = lambda m: [f"{m}_H", f"{m}_D", f"{m}_A"]
both = oos.dropna(subset=cols("bay"))
both = both.assign(**{c: (both[f"dc_{c[-1]}"] + both[f"bay_{c[-1]}"] + both[f"xgb_{c[-1]}"]) / 3
                      for c in cols("ens")})
bt = {}
for m, lab in [("dc", "Dixon-Coles"), ("bay", "Bayesian"), ("xgb", "XGBoost"), ("ens", "Ensemble"), ("mkt", "Bookmakers")]:
    p, y = both[cols(m)].values, both.y.values
    bt[lab] = dict(rps=rps(p, y).mean(), logloss=logloss(p, y).mean(),
                   acc=(p.argmax(1) == y).mean(), n=len(y))
rps_uniform = rps(np.full((len(both), 3), 1 / 3), both.y.values).mean()
cur_season = both[both.season == 2026]
bt_2026 = {lab: rps(cur_season[cols(m)].values, cur_season.y.values).mean()
           for m, lab in [("dc", "Dixon-Coles"), ("bay", "Bayesian"), ("xgb", "XGBoost"), ("ens", "Ensemble"), ("mkt", "Bookmakers")]}
bt_2026["n"] = len(cur_season)
# long-run (2022/23 - 2026/27) for the models with full coverage
long = {}
for s, gg in oos.groupby("season"):
    long[int(s)] = {lab: rps(gg[cols(m)].values, gg.y.values).mean()
                    for m, lab in [("dc", "Dixon-Coles"), ("xgb", "XGBoost"), ("mkt", "Bookmakers")]}
# bootstrap: ensemble vs market RPS difference
rng = np.random.default_rng(0)
dif = rps(both[cols("ens")].values, both.y.values) - rps(both[cols("mkt")].values, both.y.values)
boot = [rng.choice(dif, len(dif)).mean() for _ in range(4000)]
bt_ci = np.percentile(boot, [2.5, 97.5]).tolist()
# optimal weights (diagnostic only)
from scipy.optimize import minimize
P = np.stack([both[cols(m)].values for m in ["dc", "bay", "xgb"]])
def ll(w):
    w = np.exp(w) / np.exp(w).sum()
    return logloss(np.tensordot(w, P, 1), both.y.values).mean()
w_opt = minimize(ll, np.zeros(3)).x
w_opt = (np.exp(w_opt) / np.exp(w_opt).sum()).round(3).tolist()

# ----------------------------------------------------------------------------- chart 1: forecast bars
fig, ax = plt.subplots(figsize=(8.2, 3.3))
rows = ["Bookmakers", "Ensemble", "XGBoost", "Bayesian", "Dixon-Coles"]
for k, r in enumerate(rows):
    left = 0
    for key, col, lab in [("H", FCA, "Augsburg win"), ("D", DRAW, "Draw"), ("A", FCB, "Bayern win")]:
        w = summ[r][key]
        ax.barh(k, w, left=left, color=col, height=0.62 if r != "Ensemble" else 0.74,
                edgecolor=SURF, linewidth=2)
        if w > 0.055:
            ax.text(left + w / 2, k, f"{w:.0%}", ha="center", va="center",
                    color=INK if key == "D" else "white", fontsize=10)
        if k == len(rows) - 1:
            ax.text(left + w / 2 if key != "A" else left + w / 2, k + 0.55, lab, ha="center", va="bottom",
                    color=INK, fontsize=9.5)
        left += w
ax.set_yticks(range(len(rows)), [r.replace("-", "–") for r in rows])
for t in ax.get_yticklabels():
    t.set_color(INK if t.get_text() != "Ensemble" else INK)
    if t.get_text() == "Ensemble":
        t.set_fontsize(11.5)
ax.set_xlim(0, 1)
ax.set_xticks([])
ax.spines[["left", "bottom"]].set_visible(False)
ax.tick_params(left=False)
save(fig, "fig1_forecast")

# ----------------------------------------------------------------------------- chart 2: scoreline heatmap
KH, KA = 4, 7                                         # Augsburg 0-3, Bayern 0-6 (every other cell < 1%)
H = M_ens[:KH, :KA]
fig, ax = plt.subplots(figsize=(7.0, 4.0))
vmax = H.max()
cmap = matplotlib.colors.LinearSegmentedColormap.from_list("blue", BLUE_RAMP)
ax.imshow(H, cmap=cmap, vmin=0, vmax=vmax, origin="lower", aspect="auto")
for i in range(KH):
    for j in range(KA):
        v = H[i, j]
        ax.text(j, i, f"{v:.1%}" if v >= 0.005 else "", ha="center", va="center", fontsize=11,
                color="white" if v > 0.55 * vmax else INK)
mi, mj = np.unravel_index(H.argmax(), H.shape)
ax.add_patch(plt.Rectangle((mj - 0.5, mi - 0.5), 1, 1, fill=False, edgecolor=INK, linewidth=1.8))
ax.set_xticks(range(KA), [str(k) for k in range(KA)])
ax.set_yticks(range(KH), [str(k) for k in range(KH)])
ax.set_xlabel("Bayern goals")
ax.set_ylabel("Augsburg goals")
ax.spines[["left", "bottom"]].set_visible(False)
ax.tick_params(length=0, labelsize=11)
save(fig, "fig2_scorelines")

# ----------------------------------------------------------------------------- chart 3: goals per team
fig, axes = plt.subplots(1, 2, figsize=(8.2, 2.9), sharey=True)
for ax, marg, col, lab, xg in [(axes[0], M_ens.sum(1), FCA, "Augsburg goals", summ["Ensemble"]["xg_h"]),
                               (axes[1], M_ens.sum(0), FCB, "Bayern goals", summ["Ensemble"]["xg_a"])]:
    vals = list(marg[:6]) + [marg[6:].sum()]
    ax.bar(range(7), vals, color=col, width=0.72, edgecolor=SURF, linewidth=2)
    for k, v in enumerate(vals):
        ax.text(k, v + 0.008, f"{v:.0%}" if v >= 0.005 else "<1%", ha="center", va="bottom", fontsize=9, color=INK)
    ax.set_ylim(0, 0.43)
    ax.set_xticks(range(7), ["0", "1", "2", "3", "4", "5", "6+"])
    ax.set_title(f"{lab}  ·  expected {xg:.2f}", fontsize=10.5)
    ax.set_yticks([])
    ax.spines[["left"]].set_visible(False)
    ax.tick_params(length=0)
save(fig, "fig3_goals")

# ----------------------------------------------------------------------------- chart 4: strength trajectories
tid = META["tid"]
_m = pd.read_csv("data/matches.csv")
d1_sets = {s_: [tid[t] for t in set(g_.home) if t in tid] for s_, g_ in _m[_m["div"] == "D1"].groupby("season")}


def vs_d1_average(arr):
    """Re-express latent strengths relative to the average 1. Bundesliga club of that season (per draw)."""
    out = arr.copy()
    for p_ in range(arr.shape[2]):
        ids = d1_sets.get(META["first_season"] + p_ // 2)
        if ids:
            out[:, :, p_] = arr[:, :, p_] - arr[:, ids, p_].mean(axis=1, keepdims=True)
    return out


ATT_D1, DEF_D1 = vs_d1_average(ST["att"]), vs_d1_average(ST["def_"])
P_now = (2026 - META["first_season"]) * 2 + 0
per = np.arange(P_now + 1)
labels = [f"{(META['first_season'] + p // 2) % 100:02d}/{(META['first_season'] + p // 2 + 1) % 100:02d}{' H1' if p % 2 == 0 else ' H2'}" for p in per]
fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.5), sharey=False)
for ax, arr, title, sign in [(axes[0], ATT_D1, "Shots on target created (× Bundesliga average)", 1),
                             (axes[1], DEF_D1, "Shots on target conceded (× Bundesliga average)", -1)]:
    for team, col, lab in [(AWAY, FCB, "Bayern"), (HOME, FCA, "Augsburg")]:
        x = np.exp(sign * arr[:, tid[team], :P_now + 1])
        lo, mid, hi = np.percentile(x, [10, 50, 90], axis=0)
        ax.fill_between(per, lo, hi, color=col, alpha=0.14, linewidth=0)
        ax.plot(per, mid, color=col, linewidth=2)
        ax.text(per[-1] + 0.25, mid[-1], lab, color=INK, va="center", fontsize=9.5)
    ax.axhline(1, color=GRID, linewidth=1, zorder=0)
    ax.set_title(title, fontsize=10)
    ax.set_xticks(per[::2], [l.replace(" H1", "") for l in labels[::2]], fontsize=9.5)
    ax.set_xlim(-0.3, per[-1] + 1.6)
    ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:.1f}×"))
axes[1].invert_yaxis()
save(fig, "fig4_strength")

# ----------------------------------------------------------------------------- chart 5: league map
cur = sorted(set(pd.read_csv("data/matches.csv").query("season == 2026 and div == 'D1'").home))
att = np.exp(ATT_D1[:, [tid[t] for t in cur], P_now]).mean(0)
dfn = np.exp(-DEF_D1[:, [tid[t] for t in cur], P_now]).mean(0)
short = {"Bayern Munich": "Bayern", "Ein Frankfurt": "Frankfurt", "M'gladbach": "Gladbach", "Werder Bremen": "Bremen",
         "Union Berlin": "Union", "RB Leipzig": "Leipzig", "FC Koln": "Köln", "Schalke 04": "Schalke"}
from adjustText import adjust_text
fig, ax = plt.subplots(figsize=(7.4, 5.4))
texts = []
for t, x, y in zip(cur, att, dfn):
    col = FCB if t == AWAY else FCA if t == HOME else NEUTRAL
    ax.scatter(x, y, s=70 if col != NEUTRAL else 44, color=col, edgecolor=SURF, linewidth=2, zorder=3)
    texts.append(ax.text(x, y, short.get(t, t), fontsize=9, color=INK if col != NEUTRAL else MUTED))
adjust_text(texts, ax=ax, expand=(1.25, 1.5), arrowprops=dict(arrowstyle="-", color=GRID, lw=0.8))
ax.axvline(1, color=GRID, zorder=0)
ax.axhline(1, color=GRID, zorder=0)
ax.invert_yaxis()
ax.set_xlabel("Shots on target created (× average)  →  better")
ax.set_ylabel("Shots on target conceded (× average)  ↑ fewer")
ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:.1f}×"))
ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:.1f}×"))
save(fig, "fig5_league_map")

# ----------------------------------------------------------------------------- chart 6: backtest
# dot plot (not bars): the axis is zoomed in, and bars would need a zero baseline
fig, ax = plt.subplots(figsize=(7.6, 2.9))
order = ["Bookmakers", "Ensemble", "Dixon-Coles", "Bayesian", "XGBoost"]
vals = [bt[o]["rps"] for o in order]
cols_ = [MUTED if o == "Bookmakers" else ("#256abf" if o == "Ensemble" else "#6da7ec") for o in order]
for k, (v, c) in enumerate(zip(vals, cols_)):
    ax.plot([min(vals) - 0.003, v], [k, k], color=GRID, linewidth=1, zorder=1)
    ax.scatter(v, k, s=90 if order[k] in ("Ensemble", "Bookmakers") else 60, color=c,
               edgecolor=SURF, linewidth=2, zorder=3)
    ax.text(v + 0.00035, k, f"{v:.4f}", va="center", fontsize=9.5, color=INK)
ax.set_xlim(min(vals) - 0.003, max(vals) + 0.003)
ax.set_yticks(range(len(order)), [o.replace("-", "–") for o in order])
ax.invert_yaxis()
ax.set_xlabel(f"Ranked probability score, {bt['Ensemble']['n']} out-of-sample matches (lower is better)")
ax.spines[["left"]].set_visible(False)
ax.tick_params(left=False)
save(fig, "fig6_backtest")

# ----------------------------------------------------------------------------- chart 7: calibration
P_all = np.concatenate([both[c].values for c in cols("ens")])
Y_all = np.concatenate([(both.y.values == k).astype(float) for k in range(3)])
bins = np.array([0, .1, .2, .3, .4, .5, .6, .7, .85, 1.0])
idx = np.digitize(P_all, bins) - 1
cx, cy, cn = [], [], []
for b in range(len(bins) - 1):
    m = idx == b
    if m.sum() >= 15:
        cx.append(P_all[m].mean()); cy.append(Y_all[m].mean()); cn.append(m.sum())
hi_mask = P_all >= 0.6
calib_hi = dict(n=int(hi_mask.sum()), forecast=float(P_all[hi_mask].mean()), hit=float(Y_all[hi_mask].mean()))
fig, ax = plt.subplots(figsize=(4.6, 4.4))
ax.plot([0, 1], [0, 1], color=GRID, linewidth=1.2, zorder=0)
ax.plot(cx, cy, color="#256abf", linewidth=2, zorder=2)
ax.scatter(cx, cy, s=[max(18, n / 4) for n in cn], color="#256abf", edgecolor=SURF, linewidth=2, zorder=3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel("Forecast probability")
ax.set_ylabel("How often it happened")
ax.text(0.62, 0.36, "perfect calibration\nlies on the diagonal", color=MUTED, fontsize=9)
ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
save(fig, "fig7_calibration")

# ----------------------------------------------------------------------------- chart 8: XGBoost drivers
names = {"elo_h_own": "Augsburg Elo", "elo_a_own": "Bayern Elo", "elo_diff": "Elo gap",
         "h_gf_ewm": "Augsburg goals scored (form)", "h_ga_ewm": "Augsburg goals conceded (form)",
         "a_gf_ewm": "Bayern goals scored (form)", "a_ga_ewm": "Bayern goals conceded (form)",
         "h_tf_ewm": "Augsburg shots on target (form)", "h_ta_ewm": "Augsburg shots on target faced",
         "a_tf_ewm": "Bayern shots on target (form)", "a_ta_ewm": "Bayern shots on target faced",
         "h_sf_ewm": "Augsburg shots (form)", "h_sa_ewm": "Augsburg shots faced",
         "a_sf_ewm": "Bayern shots (form)", "a_sa_ewm": "Bayern shots faced",
         "d_gf": "Goals-scored gap", "d_ga": "Goals-conceded gap", "d_tf": "Shots-on-target gap",
         "d_ta": "Shots-on-target-faced gap", "h_conv": "Augsburg finishing rate", "a_conv": "Bayern finishing rate",
         "h_keep": "Augsburg save-proofness", "a_keep": "Bayern save-proofness",
         "h_pts_last5": "Augsburg points, last 5", "a_pts_last5": "Bayern points, last 5",
         "h_rest_days": "Augsburg rest days", "a_rest_days": "Bayern rest days",
         "h_promoted": "Augsburg promoted", "a_promoted": "Bayern promoted", "ghost": "Empty stadium"}
fig, axes = plt.subplots(2, 1, figsize=(7.4, 6.2))
for ax, contrib, title in [(axes[0], fd["shap_h"], "Augsburg's expected goals"), (axes[1], fd["shap_a"], "Bayern's expected goals")]:
    c = np.asarray(contrib[:-1])
    top = np.argsort(np.abs(c))[::-1][:6][::-1]
    v = c[top]
    ax.barh(range(len(top)), v, color=[FCB if x > 0 and "Bayern" in title else FCA if x > 0 else "#9ec5f4" for x in v],
            height=0.6, edgecolor=SURF, linewidth=2)
    ax.set_yticks(range(len(top)), [names.get(fd["features"][t], fd["features"][t]) for t in top], fontsize=10.5)
    ax.axvline(0, color=MUTED, linewidth=0.8)
    ax.set_title("What moves " + title, fontsize=11)
    ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"×{np.exp(v):.2f}"))
    ax.spines[["left"]].set_visible(False)
    ax.tick_params(left=False)
plt.tight_layout()
save(fig, "fig8_xgb_drivers")

# ----------------------------------------------------------------------------- numbers for the page
res = dict(summary=summ, extra=extra, backtest=bt, backtest_long=long, ens_vs_mkt_ci=bt_ci, w_opt=w_opt,
           rps_uniform=rps_uniform, backtest_2026=bt_2026, calib_hi=calib_hi,
           bayes_hyper={k: float(np.mean(S[k])) for k in ["home", "home_ghost", "gamma_home", "red_eff", "phi", "s_within", "s_summer"]},
           league_map={t: (float(a), float(d)) for t, a, d in zip(cur, att, dfn)})
json.dump(res, open("data/results.json", "w"), indent=1, default=float)
print(json.dumps({k: res[k] for k in ["summary", "extra", "backtest", "ens_vs_mkt_ci", "w_opt", "bayes_hyper"]}, indent=1, default=lambda x: round(float(x), 4)))

In [ ]:
!python scripts/analysis.py figs

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("figs/*.png")):
    print(f); display(Image(f, width=760))